In [ ]:
!nvidia-smi

In [ ]:
!apt-get -qq update
!apt-get -qq install -y build-essential cmake ninja-build

In [ ]:
!pip -q install -U scikit-build-core cmake ninja pybind11

In [ ]:
!which nvcc || echo "nvcc NOT found"
!ls -la /usr/local/cuda/bin/nvcc 2>/dev/null || true

In [ ]:
!ldconfig -p | grep -E "libcuda\.so" || true
!find /usr -name "libcuda.so*" 2>/dev/null | head

In [ ]:
!CMAKE_ARGS="-DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=60 -G Ninja \
-DCUDAToolkit_ROOT=/usr/local/cuda \
-DCMAKE_LIBRARY_PATH=/usr/local/nvidia/lib64" \
FORCE_CMAKE=1 \
pip install -U --no-binary=:all: llama-cpp-python --no-build-isolation

In [ ]:
# CPU version
# !pip -q install -U llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu125

In [ ]:
import llama_cpp
from llama_cpp import llama_cpp as lc

print("llama-cpp-python version:", getattr(llama_cpp, "__version__", "unknown"))
print("supports_gpu_offload:", lc.llama_supports_gpu_offload())

In [ ]:
import jinja2

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df = pd.read_csv("/kaggle/input/unlp-local-tests/dev_questions.csv")

In [ ]:
df.head()

In [ ]:
from huggingface_hub import hf_hub_download

repo_id = "lapa-llm/lapa-v0.1.2-instruct-GGUF"
filename = "lapa-v0.1.2-instruct-Q4_K_M.gguf"  # try Q4_K_M first

model_path = hf_hub_download(repo_id=repo_id, filename=filename)
print(model_path)

In [ ]:
llm = None
del llm

In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_ctx=2 ** 14,  # 14 = 16k ~ 14 Gi VRAM
    n_gpu_layers=-1,
    n_batch=512,
    n_threads=4,
    verbose=False,
)

def predict(prompt, max_tokens=256):
    resp = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": "You are a helpful assistant at a test. Answer with one letter."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
        max_tokens=max_tokens,
    )
    return resp["choices"][0]["message"]["content"]



In [ ]:
import time

t_start = time.time()

prompt = """
Як рекомендовано приймати ретаболіл дорослим?
A внутрішньо
B підшкірно
C орально
D внутрішньовенно
E внутрішньом’язово
F інгаляційно
"""

print(predict(prompt))

t_end = time.time()
print("time taken: ", t_end - t_start)

In [ ]:
df.head(1)

In [ ]:
QUESTION_TEMPLATE = jinja2.Template("""
{{question}}
A {{a}}
B {{b}}
C {{c}}
D {{d}}
E {{e}}
F {{f}}

Answer:
{%- if answer %}
{{answer}}
{% endif %}
""")

In [ ]:
def get_question(**kwargs) -> str:
    return QUESTION_TEMPLATE.render(**kwargs)

In [ ]:
from tqdm import tqdm

tqdm.pandas()

In [ ]:
answers = []
# for i, row in tqdm(df.head(1).iterrows(), total=len(df)):
for i, row in tqdm(df.iterrows(), total=len(df)):
    question = get_question(
        question=row["Question"],
        a=[row["A"]][0],
        b=[row["B"]][0],
        c=[row["C"]][0],
        d=[row["D"]][0],
        e=[row["E"]][0],
        f=[row["F"]][0],
    )

    pred = predict(question)
    
    # print(question)
    # print("Correct Answer:", row["Correct_Answer"])
    # print("Prediction:", pred)
    # print("\n\n\n")

    answers.append(pred)

In [ ]:
answers

In [ ]:
question = get_question(
    question=row["Question"],
    a=[row["A"]][0],
    b=[row["B"]][0],
    c=[row["C"]][0],
    d=[row["D"]][0],
    e=[row["E"]][0],
    f=[row["F"]][0],
)
print(question)

In [ ]:
len(answers)

In [ ]:
df["pred_answer"] = answers
df["pred_answer"].value_counts()

In [ ]:
df["answers_equal"] = df["pred_answer"] == df["Correct_Answer"]
df["answers_equal"].mean()

In [ ]:
df.groupby("Domain")["answers_equal"].mean()

In [ ]:
df.head(1)

In [ ]:
def get_few_shot_examples(
    df: pd.DataFrame, 
    question_id: int, 
    question_domain: str, 
    n_samples: int = 5,
    seed: int = 42
) -> str:
    sampled_questions = df[
        (df["Question_ID"] != question_id) &
        (df["Domain"] == question_domain)
    ].sample(n_samples, random_state=seed)

    shots = ""
    for _, row in sampled_questions.iterrows():
        shot = get_question(
            question=row["Question"],
            a=[row["A"]][0],
            b=[row["B"]][0],
            c=[row["C"]][0],
            d=[row["D"]][0],
            e=[row["E"]][0],
            f=[row["F"]][0],
            answer=row["Correct_Answer"]
        )
        shots += shot
    return shots

In [ ]:
df.head(1)

In [ ]:
print(get_few_shot_examples(df, 0, "domain_2"))

In [ ]:
row

In [ ]:
answers = []
for i, row in tqdm(df.iterrows(), total=len(df)):
    shots = get_few_shot_examples(df, row["Question_ID"], row["Domain"])
    question = get_question(
        question=row["Question"],
        a=[row["A"]][0],
        b=[row["B"]][0],
        c=[row["C"]][0],
        d=[row["D"]][0],
        e=[row["E"]][0],
        f=[row["F"]][0],
    )

    prompt = shots + question

    pred = predict(prompt)
    
    # print(question)
    # print("Correct Answer:", row["Correct_Answer"])
    # print("Prediction:", pred)
    # print("\n\n\n")

    answers.append(pred)

In [ ]:
print(prompt)

In [ ]:
df["pred_answer_shots"] = answers
df["pred_answer_shots"].value_counts()

In [ ]:
(df["pred_answer_shots"] == df["Correct_Answer"]).mean()

In [ ]:
!nvidia-smi